In [1]:
import pybullet as p
import pybullet_data

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

plane = p.loadURDF("plane.urdf")
robot = p.loadURDF("franka_panda/panda.urdf", useFixedBase=True)

In [2]:
for j in range(p.getNumJoints(robot)):
    info = p.getJointInfo(robot, j)

    print(
        "Index:", j,
        "| Joint:", info[1].decode(),
        "| Link:", info[12].decode(),
        "| Type:", info[2]
    )

Index: 0 | Joint: panda_joint1 | Link: panda_link1 | Type: 0
Index: 1 | Joint: panda_joint2 | Link: panda_link2 | Type: 0
Index: 2 | Joint: panda_joint3 | Link: panda_link3 | Type: 0
Index: 3 | Joint: panda_joint4 | Link: panda_link4 | Type: 0
Index: 4 | Joint: panda_joint5 | Link: panda_link5 | Type: 0
Index: 5 | Joint: panda_joint6 | Link: panda_link6 | Type: 0
Index: 6 | Joint: panda_joint7 | Link: panda_link7 | Type: 0
Index: 7 | Joint: panda_joint8 | Link: panda_link8 | Type: 4
Index: 8 | Joint: panda_hand_joint | Link: panda_hand | Type: 4
Index: 9 | Joint: panda_finger_joint1 | Link: panda_leftfinger | Type: 1
Index: 10 | Joint: panda_finger_joint2 | Link: panda_rightfinger | Type: 1
Index: 11 | Joint: panda_grasptarget_hand | Link: panda_grasptarget | Type: 4


In [83]:
for joint in range(7):
    p.resetJointState(robot, joint, targetValue=0, targetVelocity=0) # resets the arm to initial position

## start of robot grabing block:

In [2]:
pick_position = [0.5, 0.2, 0.025]
cube = p.loadURDF(
    "cube_small.urdf",
    basePosition=pick_position
)

In [85]:
import numpy as np
import time

arm_joints = list(range(7))
dt = 20 / 240

p.setTimeStep(dt)
p.setRealTimeSimulation(0)
p.setGravity(0, 0, -9.81)

downward = p.getQuaternionFromEuler([np.pi, 0, 0])
def wait_steps(steps):
    for _ in range(steps):
        p.stepSimulation()
        time.sleep(dt)

def move_hand(position, steps=720):
    current = np.array([
        p.getJointState(robot, j)[0]
        for j in arm_joints
    ])

    solution = p.calculateInverseKinematics( # calculates the inverse kinematics needed to get to the block as "position" coordinates
        bodyUniqueId=robot,
        endEffectorLinkIndex=11,
        targetPosition=position,
        targetOrientation=downward,
        maxNumIterations=300,
        residualThreshold=1e-5
    )

    targets = np.array(solution[:7])

    for step in range(steps):
        fraction = (step + 1) / steps
        intermediate = current + fraction * (targets - current)

        p.setJointMotorControlArray(
            bodyUniqueId=robot,
            jointIndices=arm_joints,
            controlMode=p.POSITION_CONTROL,
            targetPositions=intermediate.tolist(),
            forces=[87, 87, 87, 87, 12, 12, 12]
        )

        p.stepSimulation()
        time.sleep(dt)

    wait_steps(120)

    actual = np.array(
        p.getLinkState(
            robot, 11, computeForwardKinematics=True
        )[4]
    )
    
    

In [86]:
p.setJointMotorControlArray(
    robot,
    [9, 10],
    p.POSITION_CONTROL,
    targetPositions=[0.04, 0.04],
    forces=[20, 20]
)

wait_steps(240)

# Read the cube's actual position in case it has shifted
cube_position, _ = p.getBasePositionAndOrientation(cube)
x, y, z = cube_position

move_hand([x, y, z])